# Harmonisation Evaluation Metrics

In this section, we will explore several approaches for evaluating harmonisation performance in longitudinal and multi-site imaging data.

---

## Reliability-Based Metrics

We first examine:

- **Within-Subject Variability**
- **Subject Order Consistency**

These metrics are particularly useful for:

- test-retest datasets,
- traveling-subject datasets,
- or scenarios where little or no biological change is expected over time.

The goal is to evaluate the reliability and consistency of measurements across scanners, sites, or repeated acquisitions.

### Why this matters

If no true biological change is expected, large differences between repeated scans may indicate:

- scanner/site effects,
- measurement instability,
- or technical variability.

Reliable measurements are especially important in clinical and longitudinal studies where subtle biological effects are being investigated.

---

## Batch-Effect Evaluation

We then evaluate:

- **Additive Batch Effects**
- **Multiplicative Batch Effects**

These analyses can be applied to both:

- test-retest datasets,
- and true longitudinal datasets with expected biological change.

### What these tests assess

- **Additive effects** evaluate whether scanners/sites introduce systematic shifts in mean values.
- **Multiplicative effects** evaluate whether scanners/sites introduce scaling/variance differences.

These effects are important because many harmonisation methods, including ComBat-like approaches, are specifically designed to estimate and remove additive and multiplicative batch effects while preserving biological signal.

Together, these metrics provide complementary information about:
- measurement reliability,
- preservation of biological structure,
- and residual scanner/site-related bias after harmonisation.

---

## Simulated Harmonisation Dataset Parameters

The simulated dataset is generated using:

```python
df = simulate_harmonisation_data(
    n_subjects=n_subjects,
    n_timepoints=n_timepoints,
    n_features=n_features,
    seed=1,
    subject_mean=2000,
    subject_sd=80,
    noise_sd=20,
    scale=100
)
```

### Parameter Descriptions

| Parameter      | Description                                              |
| -------------- | -------------------------------------------------------- |
| `n_subjects`   | Number of simulated subjects/participants                |
| `n_timepoints` | Number of repeated longitudinal measurements per subject |
| `n_features`   | Number of simulated imaging features/IDPs                |
| `seed`         | Random seed used for reproducibility                     |
| `subject_mean` | Baseline magnitude of the simulated features             |
| `subject_sd`   | Biological variability between subjects                  |
| `noise_sd`     | Random measurement noise added to each observation       |
| `scale_value`        | Magnitude of the simulated site/scanner batch effect     |

### Simulation Model

Each observed feature value is generated as:

```text
Observed value
=
True subject value
+ Site/batch effect
+ Random noise
```

where:

* `subject_mean` and `subject_sd` determine the underlying biological variation
* `scale_value` controls the strength of the site/scanner effect
* `noise_sd` controls random measurement variability

### Site–Timepoint Confounding in the Simulation

In the current simulation setup, each longitudinal timepoint is assigned to a unique scanner/site. This creates a perfect confounding between timepoint and site effects, such that:

```text
TP0 → Site_A
TP1 → Site_B
TP2 → Site_C
...
```

As a result, all subjects scanned at a given timepoint are acquired on the same site/scanner.

This simulates a realistic longitudinal harmonisation scenario in which subjects are scanned repeatedly over time, but each visit occurs on a different scanner or acquisition site. Consequently, observed longitudinal differences may reflect both true biological change and systematic scanner/site effects, motivating the need for harmonisation methods.


---

## 1. Within-Subject Variability

This function quantifies variability across repeated measurements within the same subject.

For subjects with **2 measurements**, variability is computed as:

$$
\mathrm{Variability}(\%) =
\frac{|x_1 - x_2|}{\mathrm{mean}(x_1, x_2)} \times 100
$$

For subjects with **more than 2 measurements**, the function computes the **coefficient of variation (CoV)**:

$$
\mathrm{CoV}(\%) =
\frac{\mathrm{SD}(x)}{\mathrm{mean}(x)} \times 100
$$

Lower values indicate better agreement across scans/sites/scanners.

In [ ]:
# Uncomment if packages are missing
%pip install numpy pandas matplotlib statsmodels scikit-learn seaborn

In [ ]:
from block2_utils.HarmonisationEvaluation_functions import simulate_harmonisation_data

# Let's simulate 30 subjects, acquired at 2 timepoints and has 2 features

n_subjects=15
n_timepoints=2 
n_features=2
seed=1
subject_mean=2000
subject_sd=80
noise_sd=20
scale_value=100

df = simulate_harmonisation_data(
    n_subjects=n_subjects, 
    n_timepoints=n_timepoints, 
    n_features=n_features, 
    seed=seed,
    subject_mean=subject_mean, 
    subject_sd=subject_sd, 
    noise_sd=noise_sd,
    scale_value=scale_value
)
print(df.shape)
print(df.head())

In [ ]:
# Run code for calculating within-subject variability
from block2_utils.HarmonisationEvaluation_functions import WithinSubjVar_long
from block2_utils.HarmonisationEvaluation_plots import plot_WithinSubjVar

# df has columns: Subject, Timepoint, Site, features

features = [f"Feature_{i}" for i in range(1, n_features + 1)]

idp_matrix = df[features].to_numpy()
subjects = df["Subject"].to_list()
timepoints = df["Timepoint"].to_list()

# Get within subject variability
result = WithinSubjVar_long(
    idp_matrix=idp_matrix,
    subjects=subjects,
    timepoints=timepoints,
    idp_names=features,
)

# Plot within subject variability
plot_WithinSubjVar(
    result,
    subject_col="subject",
    limit_subjects=n_subjects,
    limit_idps_for_legend=n_features,
)

### Interpretation of Results

The output shows the within-subject variability (%) for each subject. What these plots are showing you -

- *Per-feature distribution*: For each feature (x-axis) and subject (see legend) we plot within-subject variability (%).
- *Per-feature mean*: For each feature (legend), we plot average WSV (%) across subjects.
- *Per-subject mean*: For each subject (legend), we plot average WSV (%) across features.

Because the simulated data represent repeated scans of the same individuals with no expected biological change, differences mainly reflect:

- scanner/site effects,
- measurement noise,
- technical variability.

Higher variability values indicate poorer agreement across repeated measurements, while lower values indicate better consistency.

In the multi-timepoint example, the metric is computed using the coefficient of variation (CoV), which summarizes variability across all repeated scans for each subject.

For harmonisation evaluation:

- High within-subject variability before harmonisation suggests strong scanner/site effects.
- Reduced variability after harmonisation indicates improved consistency across scanners/sites.

---

### Exercise: Extend the simulated data to multiple timepoints

Modify the simulated dataset by increasing number of subjects, timepoints, features etc. and run the code again.

1. Simulate a dataset with:
   - multiple subjects,
   - multiple timepoints,
   - multiple brain regions/features.

2. Run `WithinSubjVar_long()` using the simulated data.

3. Visualize the results using `plot_WithinSubjVar()`.

4. Inspect:
   - within-subject variability values per feature,
   - within-subject variability values per subject/datapoint,
   - differences across features.

#### *Questions*
- Which features pairs show the highest variability?
- Which subjects show the lowest consistency?
- What happens to the within subject variability if scanner/site effects become larger?

### Solution

You can check the solution code in [here](https://github.com/N-Nieto/OHBM2026_Educational_course_harmonization/tree/main/solutions/block02).

---

## 2. Subject Order Consistency

This function evaluates how consistently subjects retain their relative ordering across timepoints/scanners.

For each IDP, the function computes the **Spearman correlation** between measurements from two timepoints:

- high correlation → subjects maintain a similar ranking across scans
- low correlation → subject ordering changes more across scans

A permutation test is used to compare the observed correlation against correlations expected by chance.

- significant p-values indicate that the observed subject ordering is unlikely to occur randomly
- non-significant p-values suggest weaker or unstable ordering consistency

For harmonisation evaluation, strong and significant subject order consistency suggests that biological differences between subjects are preserved across scanners/timepoints.

In [ ]:
from block2_utils.HarmonisationEvaluation_functions import simulate_harmonisation_data
from block2_utils.HarmonisationEvaluation_functions import SubjectOrder_long
from block2_utils.HarmonisationEvaluation_plots import plot_SubjectOrder

from block2_utils.HarmonisationEvaluation_functions import simulate_harmonisation_data

# Let's simulate 15 subjects, acquired at 2 timepoints and has 2 features
n_subjects=15
n_timepoints=2 
n_features=2
seed=1
subject_mean=2000
subject_sd=80
noise_sd=20
scale_value=100

df = simulate_harmonisation_data(
    n_subjects=n_subjects, 
    n_timepoints=n_timepoints, 
    n_features=n_features, 
    seed=seed,
    subject_mean=subject_mean, 
    subject_sd=subject_sd, 
    noise_sd=noise_sd,
    scale_value=scale_value
)
print(df.shape)
print(df.head())

# df has columns: Subject, Timepoint, Site, Features
features = [f"Feature_{i}" for i in range(1, n_features + 1)]
idp_matrix = df[features].to_numpy()
subjects = df["Subject"].to_list()
timepoints = df["Timepoint"].to_list()

subjorder = SubjectOrder_long(
    idp_matrix=idp_matrix,
    subjects=subjects,
    timepoints=timepoints,
    idp_names=features,
    nPerm=100
)
print(subjorder)

plot_SubjectOrder(subjorder, p_correction="bonferroni", cmap="plasma", cmap_limits=[0.7,1])

---

### Exercise: Subject Order Consistency Across Timepoints

1. Simulate a dataset with:
   - multiple subjects,
   - multiple timepoints,
   - multiple brain regions/features.

2. Run `SubjectOrder_long()` using the simulated data.

3. Visualize the results using `plot_SubjectOrder()`.

4. Inspect:
   - Spearman correlation values,
   - permutation-test p-values,
   - differences across regions and timepoint pairs.

#### *Questions*
- Which timepoint pairs show the strongest consistency?
- Which regions show weaker consistency?
- What happens to the correlations if scanner/site effects become larger?
- Why is preserving subject ordering important in harmonisation?

### Solution

You can check the solution code in [here](https://github.com/N-Nieto/OHBM2026_Educational_course_harmonization/tree/main/solutions/block02).

---

In [ ]:
from block2_utils.HarmonisationEvaluation_functions import simulate_harmonisation_data
from block2_utils.HarmonisationEvaluation_functions import SubjectOrder_long
from block2_utils.HarmonisationEvaluation_plots import plot_SubjectOrder

# Let's simulate 30 subjects, acquired at 4 timepoints and has 5 features
n_subjects=30
n_timepoints=4 
n_features=5
seed=1
subject_mean=4000
subject_sd=80
noise_sd=80
scale_value=500

df = simulate_harmonisation_data(
    n_subjects=n_subjects, 
    n_timepoints=n_timepoints, 
    n_features=n_features, 
    seed=seed,
    subject_mean=subject_mean, 
    subject_sd=subject_sd, 
    noise_sd=noise_sd,
    scale_value=scale_value
)
print(df.shape)
print(df.head())

# df has columns: Subject, Timepoint, Site, Features
features = [f"Feature_{i}" for i in range(1, n_features + 1)]
idp_matrix = df[features].to_numpy()
subjects = df["Subject"].to_list()
timepoints = df["Timepoint"].to_list()

subjorder = SubjectOrder_long(
    idp_matrix=idp_matrix,
    subjects=subjects,
    timepoints=timepoints,
    idp_names=features,
    nPerm=100
)
print(subjorder)

plot_SubjectOrder(subjorder, p_correction="bonferroni", cmap="plasma", cmap_limits=[0.3,1])

---

## 3. Additive Batch Effect

This test asks whether the scanner/site (`batch`) adds a systematic shift to the measurements after accounting for covariates and subject-level random effects.

A simplified model is:

$$
y_{ij} = \beta_0 + \beta^T X_{ij} + \alpha_{b(i)} + u_i + \varepsilon_{ij}
$$

where:

- $y_{ij}$ = measurement for subject $i$ at observation $j$
- $X_{ij}$ = fixed effects / covariates
- $\alpha_{b(i)}$ = additive batch/site effect
- $u_i$ = subject-specific random effect
- $\varepsilon_{ij}$ = residual error

The function compares:

$$
\text{Full model: } y \sim X + \text{batch} + (1 | \text{subject})
$$

vs.

$$
\text{Reduced model: } y \sim X + (1 | \text{subject})
$$

A significant p-value suggests that the batch/site term explains additional variation, indicating an **additive batch effect**.

For harmonisation, this is the kind of shift that methods like ComBat are designed to remove.

---

## 4. Multiplicative Batch Effect

This test asks whether the variance differs across batches/sites after accounting for covariates and subject-level random effects.

A simplified model is:

$$
y_{ij} = \beta_0 + \beta^T X_{ij} + \alpha_{b(i)} + u_i + \varepsilon_{ij}
$$

The key difference is that the residual spread is compared across batch groups.

After fitting the mixed model, the residuals are tested with a variance test:

$$
H_0: \sigma^2_1 = \sigma^2_2 = \cdots = \sigma^2_K
$$

vs.

$$
H_1: \text{at least one batch has different variance}
$$

The function uses the **Fligner-Killeen test** on model residuals grouped by batch.

A significant p-value suggests a **multiplicative batch effect**, meaning one site/scanner has more or less variability than another.

For harmonisation, this corresponds to a scale/variance difference rather than a mean shift.

## Interpretation

- **Additive effect significant**: batches differ in mean level.
- **Multiplicative effect significant**: batches differ in variance.
- **Both significant**: scanner/site influences both location and spread.

In a longitudinal harmonisation setting, successful correction should reduce both effects while preserving subject-level structure and true biological change.

---

## Simulating longitudinal data with additive and multiplicative batch effects

This simulation generates synthetic longitudinal imaging data with:

* subject-specific biological variability
* longitudinal change over time
* age effects
* site/scanner batch effects
* random measurement noise

Unlike the previous simulation, site and timepoint are **not perfectly confounded**. Each subject can be scanned at different sites across timepoints:

```python
subject_sites = np.random.choice(sites, size=n_timepoints, replace=True)
```

This creates a another realistic harmonisation scenario.

The simulated observed value is approximately:

```text
Observed value
=
True biological signal
+ Additive site effect
+ Measurement noise
```

where:

* `additive_shift` introduces site-specific offsets
* `multiplicative_scale` changes the amount of site-specific noise
* `slope_sd` controls longitudinal subject trajectories
* `noise_sd` controls random measurement variability

This simulation is useful for studying how additive and multiplicative scanner/site effects influence longitudinal analyses and harmonisation methods.


In [ ]:
from block2_utils.HarmonisationEvaluation_functions import simulate_longitudinal_batch_data_mixed
from block2_utils.HarmonisationEvaluation_plots import plot_additive_multiplicative_effects

# Let's simulate some data: 100 subjects, 3 timepoints, 2 sites, features from 4 brain regions
n_subjects=100
n_timepoints=3
n_sites=2
n_features=4
additive_shift = {"Site_B": {"Feature_1": 600, "Feature_3": -150}}
multiplicative_scale = {"Site_B": {"Feature_2": 15.0}, "Site_A": {"Feature_4": 2.7}}

df = simulate_longitudinal_batch_data_mixed(
    n_subjects=n_subjects,
    n_timepoints=n_timepoints,
    n_sites=n_sites,
    n_features=n_features,
    additive_shift=additive_shift,
    multiplicative_scale=multiplicative_scale,
    seed=1,
)

print(df.head(10))
feature_cols = [f"Feature_{i}" for i in range(1, n_features + 1)]

# Let's visualise additive and multiplicative effects
plot_additive_multiplicative_effects(df, feature_cols=feature_cols, batch_col="Site")


In [ ]:
from block2_utils.HarmonisationEvaluation_functions import AdditiveEffect_long
from block2_utils.HarmonisationEvaluation_functions import MultiplicativeEffect_long

feature_cols = [f"Feature_{i}" for i in range(1, n_features + 1)]

# Let's estimate additive and multiplicative batch effects
additive_results, model_defs_add = AdditiveEffect_long(
    data=df,
    idp_names=feature_cols,
    idvar="Subject",
    batchvar="Site",
    timevar="Timepoint",
    fix_eff=["Age"],
    ran_eff=["Subject"],
    do_zscore=True,
    verbose=False,
)
print(additive_results)

multiplicative_results, model_defs_mul = MultiplicativeEffect_long(
    data=df,
    idp_names=feature_cols,
    idvar="Subject",
    batchvar="Site",
    timevar="Timepoint",
    fix_eff=["Age"],
    ran_eff=["Subject"],
    do_zscore=True,
    verbose=False,
)
print(multiplicative_results)


## Exercise: Identifying additive and multiplicative batch effects

Using the statistical results above, identify which simulated features exhibit:

* **Additive batch effects** (site-related shifts in mean values)
* **Multiplicative batch effects** (site-related differences in variability)

Hints:

* Significant Wald test results indicate potential additive effects
* Significant Fligner test results indicate potential multiplicative effects

Questions:

1. Which features show significant additive batch effects?
2. Which features show significant multiplicative batch effects?
3. Are there any features that show both types of batch effects?
4. Which features appear relatively stable across sites?

### Solution

You can check the solution code in [here](https://github.com/N-Nieto/OHBM2026_Educational_course_harmonization/tree/main/solutions/block02).
